In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 73.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0

In [3]:
import torch
import gc

# Clear all GPU memory before starting
gc.collect()
torch.cuda.empty_cache()

# Check memory is clean
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
    print(f"  Total:", round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2), "GB")
    print(f"  Used:", round(torch.cuda.memory_allocated(i) / 1e9, 2), "GB")
    print(f"  Free:", round((torch.cuda.get_device_properties(i).total_memory - torch.cuda.memory_allocated(i)) / 1e9, 2), "GB")

GPU 0: Tesla T4
  Total: 15.64 GB
  Used: 0.0 GB
  Free: 15.64 GB
GPU 1: Tesla T4
  Total: 15.64 GB
  Used: 0.0 GB
  Free: 15.64 GB


In [4]:
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

transformers: 5.0.0
datasets: 4.8.5
peft: 0.18.1
trl: 1.6.0
bitsandbytes: 0.49.2


In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace")

Logged in to HuggingFace


In [6]:
from datasets import load_dataset

dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

# Create validation split
split = dataset['train'].train_test_split(test_size=0.1, seed=42)
dataset['train'] = split['train']
dataset['validation'] = split['test']

# Use 3000 training examples — fits safely in 9hr Kaggle session
dataset['train'] = dataset['train'].select(range(4000))
dataset['validation'] = dataset['validation'].select(range(400))

# Keep full test set for evaluation
print("Train:", len(dataset['train']))
print("Validation:", len(dataset['validation']))
print("Test:", len(dataset['test']))

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

Train: 4000
Validation: 400
Test: 1273


In [7]:
def format_example(example):
    options_str = "\n".join([f"{k}) {v}" for k, v in example['options'].items()])
    text = f"""### Question:
{example['question']}

### Options:
{options_str}

### Answer:
{example['answer_idx']}) {example['answer']}"""
    return {"text": text}

dataset['train'] = dataset['train'].map(format_example)
dataset['validation'] = dataset['validation'].map(format_example)
dataset['test'] = dataset['test'].map(format_example)

print("Dataset formatted")
print(dataset['train'][0]['text'])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Dataset formatted
### Question:
A 60-year-old man comes to the physician because of flank pain, rash, and blood-tinged urine for 1 day. Two months ago, he was started on hydrochlorothiazide for hypertension. He takes acetaminophen for back pain. Examination shows a generalized, diffuse maculopapular rash. Serum studies show a creatinine concentration of 3.0 mg/dL. Renal ultrasonography shows no abnormalities. Which of the following findings is most likely to be observed in this patient?

### Options:
A) Dermal IgA deposition on skin biopsy
B) Crescent-shape extracapillary cell proliferation
C) Mesangial IgA deposits on renal biopsy
D) Urinary eosinophils

### Answer:
D) Urinary eosinophils


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "BioMistral/BioMistral-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded")
print("GPU memory used:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The repo does no

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded
GPU memory used: 1.61 GB


In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [10]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./medqa-biomistral",
    num_train_epochs=1,                 # 1 epochs on 4000 examples ~ 6-7 hrs
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    logging_steps=50,
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    report_to="none",
    load_best_model_at_end=True,
    max_length=512,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()
print("Training complete")


/tmp/ipykernel_22/2318674807.py:3: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,1.078639,1.059970,1.080339,0.728265,833161.000000


Training complete


In [11]:
import os

# Check checkpoint was saved locally
checkpoint_path = "./medqa-biomistral"
files = os.listdir(checkpoint_path)
print("Files in output dir:", files)

# Check which checkpoint we have
for f in files:
    full_path = os.path.join(checkpoint_path, f)
    if os.path.isdir(full_path):
        print(f"Checkpoint: {full_path}")
        print("  Files:", os.listdir(full_path))

# Confirm HF username
from huggingface_hub import whoami
user = whoami()
print("\nLogged in as:", user['name'])
print("Model will be saved to:", f"https://huggingface.co/{user['name']}/medqa-biomistral-finetuned")

Files in output dir: ['checkpoint-200', 'checkpoint-250', 'README.md']
Checkpoint: ./medqa-biomistral/checkpoint-200
  Files: ['scheduler.pt', 'chat_template.jinja', 'optimizer.pt', 'tokenizer.json', 'adapter_model.safetensors', 'training_args.bin', 'rng_state.pth', 'tokenizer_config.json', 'trainer_state.json', 'adapter_config.json', 'README.md']
Checkpoint: ./medqa-biomistral/checkpoint-250
  Files: ['scheduler.pt', 'chat_template.jinja', 'optimizer.pt', 'tokenizer.json', 'adapter_model.safetensors', 'training_args.bin', 'rng_state.pth', 'tokenizer_config.json', 'trainer_state.json', 'adapter_config.json', 'README.md']

Logged in as: SinghManas14
Model will be saved to: https://huggingface.co/SinghManas14/medqa-biomistral-finetuned


In [12]:
import time
# Wait 30 seconds to make sure training is fully complete
time.sleep(30)
print("Starting HF Hub save...")

Starting HF Hub save...


In [13]:
# Save locally first
from huggingface_hub import whoami

# Get username automatically — no hardcoding
username = whoami()['name']
repo_id = f"{username}/medqa-biomistral-finetuned"

print(f"Saving to: {repo_id}")

# Save locally
trainer.model.save_pretrained("./medqa-biomistral-final")
tokenizer.save_pretrained("./medqa-biomistral-final")

# Push to HF Hub
trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"Model saved to: https://huggingface.co/{repo_id}")

Saving to: SinghManas14/medqa-biomistral-finetuned


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Model saved to: https://huggingface.co/SinghManas14/medqa-biomistral-finetuned


In [14]:
from peft import PeftModel
from tqdm import tqdm
import gc

# Clear training memory
del trainer
torch.cuda.empty_cache()
gc.collect()

# Reload in inference mode
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "./medqa-biomistral-final")
model.eval()

# Evaluation function
def ask_model(question, options):
    options_str = "\n".join([f"{k}) {v}" for k, v in options.items()])
    prompt = f"""### Question:
{question}

### Options:
{options_str}

### Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Answer:")[-1].strip()

# Run evaluation on full test set
correct = 0
total = 0

for example in tqdm(dataset['test']):
    try:
        result = ask_model(example['question'], example['options'])
        predicted = result.strip()[0].upper()
        actual = example['answer_idx'].strip().upper()
        if predicted == actual:
            correct += 1
        total += 1
    except:
        total += 1

accuracy = correct / total * 100
print(f"\n{'='*40}")
print(f"Test examples: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Random baseline: 25.00%")
print(f"BioMistral base: ~59.00%")
print(f"Your fine-tuned model: {accuracy:.2f}%")
print(f"{'='*40}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The repo does no


Test examples: 1273
Correct: 652
Accuracy: 51.22%
Random baseline: 25.00%
BioMistral base: ~59.00%
Your fine-tuned model: 51.22%
